# 第18章　医療AIのインフラ ― クラウドとオンプレミス

**『医療診断支援AIの社会実装（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 推論を「サービス」として運ぶ ― モデルサービングとオーケストレーション

```yaml
# 推論サービスの最小骨格（抜粋）。resources と readinessProbe は container の配下に置く。
kind: Deployment
spec:
  replicas: 2                      # 冗長化。ただし別ノードへの配置は別途 topologySpread 等で指定
  template:
    spec:
      containers:
        - name: inference
          image: registry.hospital.local/pancreas-ai:v2.3  # 版を明示（ライフサイクル管理の章）
          resources:
            limits:
              nvidia.com/gpu: 1
          readinessProbe:          # モデル読込完了まで要求を回さない
            httpGet:
              path: /health
              port: 8000
```

## モデルを軽くする3つの道 ― 蒸留・枝刈り・量子化

In [ ]:
import torch
import torch.nn.functional as F

# 蒸留：教師の軟らかい確率を、生徒が温度Tで真似る
p_t = F.log_softmax(teacher(x) / T, dim=1)
p_s = F.log_softmax(student(x) / T, dim=1)
loss = alpha * T*T * F.kl_div(p_s, p_t.exp(), reduction="batchmean") \
       + (1 - alpha) * F.cross_entropy(student(x), y)   # 蒸留＋通常の正解損失

## 必要なところだけ計算する ― 混合エキスパート（MoE）

In [ ]:
import torch

# トークン/特徴ごとに上位k個のエキスパートだけを使う（1サンプル分の骨格）
scores = gate(x)                              # 各エキスパートへの適性スコア
vals, idx = scores.topk(k=2, dim=-1)          # 上位2つ（重みと番号を分けて受ける）
w = torch.softmax(vals, dim=-1)               # 選ばれた分だけで正規化
y = sum(w[..., j:j+1] * experts[idx[..., j]](x) for j in range(2))   # 重み付き和
# ※これは1サンプル分の骨格。バッチでは idx がベクトルになり ModuleList を添字できない。
#   実装ではエキスパートごとに該当サンプルを集めて通す（サンプル単位のディスパッチ）。
# 補助損失：特定エキスパートに負荷が偏らないよう均す（load balancing）
aux = load_balance_loss(scores)